In [1]:
import pandas as pd
from light_tools import bands_to_wavelengths, pull_spectral_images
from sklearn.model_selection import train_test_split
from generators import SpectralImageDataGenerator


# Load the band information
bands = pd.read_csv("enmap_bands.csv")
band_info_df = bands_to_wavelengths(bands)
wavelengths = bands['CW (nm)'].values


# Load the spectral images
spectral_images = pull_spectral_images("../HySpecNet/hyspecnet-11k/patches")

# Split the data into training and validation sets
image_paths_train, image_paths_val = train_test_split(spectral_images, test_size=0.2, random_state=42)
(len(image_paths_train), len(image_paths_val))


(977, 245)

In [2]:
import matplotlib.pyplot as plt

def plot_images_from_generator(generator, num_images=1, band_index=930):
    for i in range(num_images):
        # Retrieve a batch of images
        images_batch = generator.__getitem__(i)[0]  # Assuming the first element is the batch of images
        
        plt.figure(figsize=(20, 4))
        
        # Determine the number of images to plot (up to 10)
        num_plot_images = min(len(images_batch), 10)
        for j in range(num_plot_images):
            plt.subplot(1, num_plot_images, j + 1)
            
            # Check if the image is 2D or 3D and plot accordingly
            if images_batch[j].ndim == 3:
                # For 3D images, plot a specific band
                plt.imshow(images_batch[j][:, :, band_index], cmap='gray')
            elif images_batch[j].ndim == 2:
                # For 2D images, plot as is
                plt.imshow(images_batch[j], cmap='gray')
            else:
                raise ValueError("Unexpected image dimensionality")
                
            plt.title(f"Batch {i} - Image {j}")
            plt.axis("off")
        plt.show()


train_gen = SpectralImageDataGenerator(image_paths_train, batch_size=32)
# plot_images_from_generator(train_gen, num_images=4)



In [3]:
val_gen = SpectralImageDataGenerator(image_paths_val, batch_size=32)
# plot_images_from_generator(val_gen, num_images=4)

In [4]:
from light_tools import visualize_bands, plot_band_histograms, generate_spectral_profile, plot_spectral_profile

# images = train_gen[0]
# visualize_bands(images[0])
# # plot_band_histograms(images[0], [0, images[0].shape[2] // 2, images[0].shape[2] - 1])
# plot_band_histograms(images[0], [130,141,165])

# # take all bands
# band_info_df = bands_to_wavelengths(bands)

# # Generate a complete 'wavelengths' array for all bands
# wavelengths = bands['CW (nm)'].values

# # Generate a spectral profile for the entire image or a specific pixel
# spectral_profile = generate_spectral_profile(images[0])

# plot_spectral_profile(band_info_df, wavelengths, spectral_profile)

In [5]:
from utils import analyze_hyperspectral_data

# nan_info, min_value, max_value = analyze_hyperspectral_data(train_gen, plot_nans=True)
#(nan_info, min_value, max_value)

In [6]:
from net_def import create_3d_autoencoder

# Define the model
input_shape = (128, 128, 224, 1)  # Example input shape, adjust as necessary
autoencoder = create_3d_autoencoder(input_shape)
autoencoder.compile(optimizer='adam', loss='mean_squared_error')

# For feature extraction, compile the model with no loss function as we're not training it

autoencoder.summary()

2.15.0
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 128, 128, 224,    0         
                             1)]                                 
                                                                 
 conv3d (Conv3D)             (None, 128, 128, 224, 3   896       
                             2)                                  
                                                                 
 max_pooling3d (MaxPooling3  (None, 64, 64, 112, 32)   0         
 D)                                                              
                                                                 
 conv3d_1 (Conv3D)           (None, 64, 64, 112, 16)   13840     
                                                                 
 max_pooling3d_1 (MaxPoolin  (None, 32, 32, 56, 16)    0         
 g3D)                                                 

2024-03-12 16:35:13.534526: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3 Max
2024-03-12 16:35:13.534552: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 128.00 GB
2024-03-12 16:35:13.534556: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 48.00 GB
2024-03-12 16:35:13.534601: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2024-03-12 16:35:13.534634: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [7]:
autoencoder.fit(
    train_gen,  # The generator provides both x and y
    epochs=50,
    # batch_size=64,  # Make sure this matches your generator's batch size or remove it since the generator dictates the batch size
    # shuffle=True,  # This should be handled by your generator
    validation_data=val_gen,  # Assuming val_gen is similarly structured
)


Epoch 1/50


2024-03-12 16:35:22.242154: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


In [ ]:
# from net_def import create_cnn
# from keras.optimizers import Adam
# from keras.losses import MeanSquaredError

# # Define the CNN model for feature extraction
# input_shape = (128, 128, 224)  # The shape of your hyperspectral images
# model = create_cnn(input_shape)

# # For feature extraction, compile the model with no loss function as we're not training it
# model.compile(optimizer=Adam(), loss=MeanSquaredError())
# model.summary()


In [ ]:
# # Instantiate the data generator
# train_gen = SpectralImageDataGenerator(image_paths_train, batch_size=32)
# val_gen = SpectralImageDataGenerator(image_paths_val, batch_size=32)

# # If extracting features, run the model prediction on the data generator
# # This does not train the model; it only runs the forward pass to extract features
# features = model.predict(train_gen, verbose=1)
# features